# Task 2.5: LightGBM Boundary Rejection Filter
**Owner:** Marco | **Module:** Anomaly-Aware Candidate Boundary Pruning

### Objectives:
1. **Mock Stratified 80/20 Split:** Create clean local partitions simulating Yi Kai's upcoming deliverable.
   - 🟢 Train Normal (80% Normal only) for generative models (GMM / TabDDPM).
   - 🟢🔴 Val Split (20% Normal + Anomaly) strictly for training & evaluating this LightGBM boundary filter.
2. **Boundary Estimator (LightGBM):** Train gradient-boosted decision trees using native categorical feature support (no one-hot expansion) to estimate (\text{anomaly} \mid \tilde{x})$.
3. **Candidate Filtering Engine:** Prune synthetic candidates where (y=1 \mid \tilde{x}) > \tau_{\text{reject}}$ to prevent decision boundary dilation.
4. **Threshold Sensitivity Sweep:** Evaluate threshold sweep across $\tau_{\text{reject}} \in [0.05, 0.20]$.
5. **Modular Export:** Export clean  model artifacts and reusable Python module for Tasks 3, 4, and 5 plug-and-play.

In [ ]:
import os
import sys
import pickle
import numpy as np
import pandas as pd
import lightgbm as lgb
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score, classification_report, roc_curve, precision_recall_curve

# Configure paths
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.basename(os.getcwd()) == "tmp_notebook" else os.path.abspath(os.getcwd())
DATA_DIR = os.path.join(REPO_ROOT, "data")
RESULTS_DIR = os.path.join(REPO_ROOT, "results")

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
print(f"Project root: {REPO_ROOT}")
print(f"LightGBM version: {lgb.__version__}")

## 1. Mock Stratified 80/20 Data Split

Until Yi Kai delivers the finalized preprocessing on 12 Oct, we create a reproducible mock split directly from  (or the project root).

**Rule Enforcement:**
- **Train (80%):** Discard anomalies $\to$ save **🟢 Normal Only** for GMM / TabDDPM.
- **Val (20%):** Retain both $\to$ save **🟢🔴 Normal + Anomaly** for LightGBM filter.

In [ ]:
def prepare_mock_splits(raw_csv_path, dataset_name, test_size=0.20, seed=42):
    df = pd.read_csv(raw_csv_path)
    assert "is_anomaly" in df.columns, f"Target 'is_anomaly' missing in {raw_csv_path}"
    
    train_df, val_df = train_test_split(
        df,
        test_size=test_size,
        random_state=seed,
        stratify=df["is_anomaly"]
    )
    
    # Train split: keep NORMAL ONLY
    train_normal = train_df[train_df["is_anomaly"] == 0].drop(columns=["is_anomaly"]).reset_index(drop=True)
    val_clean = val_df.reset_index(drop=True)
    
    out_dir = os.path.join(DATA_DIR, dataset_name)
    os.makedirs(out_dir, exist_ok=True)
    
    train_normal_path = os.path.join(out_dir, "train_normal_mock.csv")
    val_path = os.path.join(out_dir, "val_split_mock.csv")
    
    train_normal.to_csv(train_normal_path, index=False)
    val_clean.to_csv(val_path, index=False)
    
    print(f"[{dataset_name}] Total: {len(df)} rows")
    print(f"  - 🟢 Train Normal (Mock for Gen Models): {len(train_normal)} rows -> {train_normal_path}")
    print(f"  - 🟢🔴 Val Split (For LightGBM Filter): {len(val_clean)} rows (Normal: {(val_clean['is_anomaly']==0).sum()}, Anomaly: {(val_clean['is_anomaly']==1).sum()}) -> {val_path}")
    
    return train_normal, val_clean

# Locate raw datasets (handles nested repo structure)
project_parent = os.path.abspath(os.path.join(REPO_ROOT, ".."))
nsl_raw = os.path.join(REPO_ROOT, "Dataset", "NSL-KDD", "train.csv") if os.path.exists(os.path.join(REPO_ROOT, "Dataset", "NSL-KDD", "train.csv")) else os.path.join(project_parent, "Dataset", "NSL-KDD", "train.csv")
unsw_raw = os.path.join(REPO_ROOT, "Dataset", "UNSW-NB15", "train.csv") if os.path.exists(os.path.join(REPO_ROOT, "Dataset", "UNSW-NB15", "train.csv")) else os.path.join(project_parent, "Dataset", "UNSW-NB15", "train.csv")

_, nsl_val = prepare_mock_splits(nsl_raw, "NSL-KDD", seed=RANDOM_SEED)
_, unsw_val = prepare_mock_splits(unsw_raw, "UNSW-NB15", seed=RANDOM_SEED)

## 2. Universal Schema Inspector

LightGBM supports native categorical handling via Pandas  dtype. This completely avoids one-hot dimensionality explosion (e.g. 67 service levels in NSL-KDD, 86 proto levels in UNSW-NB15) while learning precise split points.

In [ ]:
def detect_feature_types(df, target_col="is_anomaly"):
    feature_cols = [c for c in df.columns if c != target_col and c != "id"]
    cat_cols = []
    num_cols = []
    
    for col in feature_cols:
        dtype = df[col].dtype
        if (
            isinstance(dtype, pd.CategoricalDtype)
            or pd.api.types.is_string_dtype(dtype)
            or pd.api.types.is_object_dtype(dtype)
        ):
            cat_cols.append(col)
        else:
            num_cols.append(col)
            
    return feature_cols, cat_cols, num_cols

for name, df_val in [("NSL-KDD", nsl_val), ("UNSW-NB15", unsw_val)]:
    features, cats, nums = detect_feature_types(df_val)
    print(f"[{name}] Total Features: {len(features)} | Categorical: {len(cats)} ({cats}) | Numerical: {len(nums)}")


## 3. LightGBM Boundary Estimator Implementation

We encapsulate the filter in a clean, reusable `LightGBMBoundaryFilter` class with:
- Native categorical encoding.
- Balanced class weighting (`class_weight="balanced"`).
- Candidate pruning by threshold $\tau_{\text{reject}}$: discard candidate $\tilde{x}$ if $P(\text{anomaly} \mid \tilde{x}) > \tau_{\text{reject}}$.


In [ ]:
class LightGBMBoundaryFilter:
    """
    LightGBM-based Anomaly Boundary Estimator for Synthetic Candidate Filtering.
    """
    def __init__(self, cat_cols=None, params=None, random_state=42):
        self.cat_cols = cat_cols or []
        self.random_state = random_state
        self.default_params = {
            "objective": "binary",
            "boosting_type": "gbdt",
            "n_estimators": 300,
            "learning_rate": 0.05,
            "num_leaves": 31,
            "class_weight": "balanced",
            "random_state": self.random_state,
            "verbosity": -1,
            "n_jobs": -1
        }
        if params:
            self.default_params.update(params)
        self.model = lgb.LGBMClassifier(**self.default_params)
        self.feature_names = []
        self.cat_dtypes = {}

    def _prepare_features(self, df):
        X = df.copy()
        if "id" in X.columns:
            X = X.drop(columns=["id"])
        if "is_anomaly" in X.columns:
            X = X.drop(columns=["is_anomaly"])
            
        for col in self.cat_cols:
            if col in X.columns:
                X[col] = X[col].astype(str).astype("category")
        return X

    def fit(self, val_df, target_col="is_anomaly"):
        self.feature_names, self.cat_cols, _ = detect_feature_types(val_df, target_col)
        X = self._prepare_features(val_df)
        y = val_df[target_col].values
        
        # Record category levels for consistent evaluation
        for col in self.cat_cols:
            self.cat_dtypes[col] = X[col].dtype
            
        self.model.fit(X, y)
        return self

    def predict_anomaly_prob(self, df_candidates):
        X = self._prepare_features(df_candidates)
        # Ensure all columns present
        X = X[self.feature_names]
        return self.model.predict_proba(X)[:, 1]

    def filter_candidates(self, df_candidates, tau_reject=0.10):
        """
        Prunes candidates whose predicted anomaly probability exceeds tau_reject.
        Returns: (filtered_df, survival_rate, anomaly_probabilities)
        """
        probs = self.predict_anomaly_prob(df_candidates)
        keep_mask = probs <= tau_reject
        filtered_df = df_candidates[keep_mask].copy()
        survival_rate = float(keep_mask.sum()) / max(len(df_candidates), 1)
        return filtered_df, survival_rate, probs

    def save(self, filepath):
        with open(filepath, "wb") as f:
            pickle.dump(self, f)
        print(f"BoundaryFilter saved to {filepath}")

    @staticmethod
    def load(filepath):
        with open(filepath, "rb") as f:
            obj = pickle.load(f)
        print(f"BoundaryFilter loaded from {filepath}")
        return obj

## 4. Stratified 5-Fold Cross-Validation on the 20% Val Split

We evaluate the LightGBM boundary model on the 20% validation split using 5-fold cross-validation to ensure strong generalization without overfitting.

In [ ]:
def evaluate_filter_cv(val_df, dataset_name, n_splits=5):
    features, cats, _ = detect_feature_types(val_df)
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_SEED)
    
    y = val_df["is_anomaly"].values
    oof_probs = np.zeros(len(val_df))
    
    for fold, (train_idx, eval_idx) in enumerate(skf.split(val_df, y)):
        fold_train = val_df.iloc[train_idx]
        fold_eval = val_df.iloc[eval_idx]
        
        bf = LightGBMBoundaryFilter(cat_cols=cats, random_state=RANDOM_SEED + fold)
        bf.fit(fold_train)
        oof_probs[eval_idx] = bf.predict_anomaly_prob(fold_eval)
        
    roc_auc = roc_auc_score(y, oof_probs)
    pr_auc = average_precision_score(y, oof_probs)
    print(f"========================================")
    print(f"[{dataset_name}] 5-Fold OOF Boundary Estimator Metrics:")
    print(f"  ROC-AUC: {roc_auc:.4f}")
    print(f"  PR-AUC (Anomaly Precision-Recall): {pr_auc:.4f}")
    print(f"========================================
")
    return y, oof_probs

nsl_y, nsl_oof_probs = evaluate_filter_cv(nsl_val, "NSL-KDD")
unsw_y, unsw_oof_probs = evaluate_filter_cv(unsw_val, "UNSW-NB15")

## 5. Threshold Sensitivity Analysis ($\tau_{\text{reject}} \in [0.05, 0.25]$)

In our proposal (Section 5.3), we stated that $\tau_{\text{reject}}$ will be calibrated to govern how aggressively candidate points near attack boundaries are pruned.
Here we measure:
- **Normal Retention Rate:** Percentage of legitimate normal points preserved.
- **Attack Leakage Rate:** Percentage of attack-like boundary points permitted to leak.

In [ ]:
def analyze_thresholds(y_true, probs, dataset_name):
    thresholds = [0.02, 0.05, 0.08, 0.10, 0.12, 0.15, 0.20, 0.25, 0.30]
    records = []
    
    for tau in thresholds:
        accepted_mask = probs <= tau
        
        normal_mask = (y_true == 0)
        attack_mask = (y_true == 1)
        
        normal_retention = accepted_mask[normal_mask].mean() * 100
        attack_leakage = accepted_mask[attack_mask].mean() * 100
        
        records.append({
            "tau_reject": tau,
            "Normal_Retention_%": round(normal_retention, 2),
            "Attack_Leakage_%": round(attack_leakage, 2),
            "Attack_Rejected_%": round(100.0 - attack_leakage, 2)
        })
        
    res_df = pd.DataFrame(records)
    print(f"[{dataset_name}] Threshold Sweep Results:")
    print(res_df.to_string(index=False))
    return res_df

nsl_thresh_df = analyze_thresholds(nsl_y, nsl_oof_probs, "NSL-KDD")
print()
unsw_thresh_df = analyze_thresholds(unsw_y, unsw_oof_probs, "UNSW-NB15")

## 6. Plotting ROC & Precision-Recall Boundary Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for name, y_true, probs in [("NSL-KDD", nsl_y, nsl_oof_probs), ("UNSW-NB15", unsw_y, unsw_oof_probs)]:
    fpr, tpr, _ = roc_curve(y_true, probs)
    prec, rec, _ = precision_recall_curve(y_true, probs)
    roc_score = roc_auc_score(y_true, probs)
    pr_score = average_precision_score(y_true, probs)
    
    axes[0].plot(fpr, tpr, label=f"{name} (AUC={roc_score:.3f})")
    axes[1].plot(rec, prec, label=f"{name} (PR-AUC={pr_score:.3f})")

axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.5)
axes[0].set_title("ROC Curve (Boundary Discriminator)")
axes[0].set_xlabel("False Positive Rate")
axes[0].set_ylabel("True Positive Rate")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].set_title("Precision-Recall Curve (Attack Discrimination)")
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plot_path = os.path.join(RESULTS_DIR, "boundary_filter_performance.png")
plt.savefig(plot_path, dpi=150)
plt.show()
print(f"Performance plot saved to {plot_path}")

## 7. Model Export for Tasks 3, 4, and 5 (Plug & Play)

We fit the final boundary filters on the full 20% validation split for each dataset and export them to .
Diana and Yi Kai can load them with a single line: .

In [ ]:
MODELS_DIR = os.path.join(RESULTS_DIR, "models")
os.makedirs(MODELS_DIR, exist_ok=True)

# Train & export NSL-KDD filter
nsl_filter = LightGBMBoundaryFilter(random_state=RANDOM_SEED)
nsl_filter.fit(nsl_val)
nsl_model_path = os.path.join(MODELS_DIR, "boundary_filter_nsl_kdd.pkl")
nsl_filter.save(nsl_model_path)

# Train & export UNSW-NB15 filter
unsw_filter = LightGBMBoundaryFilter(random_state=RANDOM_SEED)
unsw_filter.fit(unsw_val)
unsw_model_path = os.path.join(MODELS_DIR, "boundary_filter_unsw_nb15.pkl")
unsw_filter.save(unsw_model_path)

print("\n[SUCCESS] Plug-and-play filter models exported ready for candidate generation!")